In [ ]:
import sqlite3

# --- 1. DATABASE SETUP ---
conn = sqlite3.connect("tickets.db", check_same_thread=False)
cursor = conn.cursor()

# Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS tickets (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    source TEXT,
    destination TEXT,
    price REAL
)
""")

# Seed data
cursor.execute("DELETE FROM tickets")

data = [
    ("FlightAI", "Mumbai", "Delhi", 5000),
    ("FlightAI", "Mumbai", "Tokyo", 45000),
    ("FlightAI", "Delhi", "Bangalore", 4000),
]

cursor.executemany(
    "INSERT INTO tickets (name, source, destination, price) VALUES (?, ?, ?, ?)",
    data
)

conn.commit()

In [ ]:
def normalize(text): return str(text).strip().title()

def get_price(source, destination):
    cursor.execute("SELECT name, price FROM tickets WHERE source=? AND destination=?", (normalize(source), normalize(destination)))
    return cursor.fetchall()

def book_ticket(name, source, destination):
    res = get_price(source, destination)
    if not res: return f"Error: No flights from {source} to {destination}."
    return f"SUCCESS: Booked {name} on {res[0][0]} for ₹{res[0][1]}"

def get_all_flights():
    """Returns a formatted string of all available routes and prices."""
    cursor.execute("SELECT source, destination, price FROM tickets")
    rows = cursor.fetchall()
    if not rows:
        return "No flights available in the database."
    
    lines = [f"✈️ {src} to {dst}: ₹{price}" for src, dst, price in rows]
    return "\n".join(lines)

In [ ]:
from openai import OpenAI
import json
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

system_prompt = """
You are a Ticket Booking Bot. 
You MUST speak ONLY in JSON. NO COMMENTS. NO CHAT.

FORMAT 1 (Tool): {"tool": "book_ticket", "args": {"name": "Jon", "source": "Mumbai", "destination": "Tokyo"}}
FORMAT 2 (Final): {"final": "I have booked your ticket!"}

ALLOWED CITIES: Mumbai, Delhi, Tokyo, Bangalore.
If you are missing the user's Name or Source, ASK FOR IT using the 'final' format.
"""




In [ ]:
def agent_with_logs(user_input):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_input}]
    thought_log = []
    
    for i in range(4): 
        try:
            response = ollama.chat.completions.create(model="llama3.2:1b", messages=messages)
            reply = response.choices[0].message.content.strip()
            
            thought_log.append(f"🤖 Step {i+1} RAW RESPONSE:\n{reply}\n{'-'*20}")
            
            # Clean comments and extract JSON
            reply_clean = re.sub(r"//.*", "", reply)
            match = re.search(r"\{.*\}", reply_clean, re.DOTALL)
            
            if not match: 
                thought_log.append(f"⚠️ Step {i+1}: No JSON found in raw text. Retrying...")
                messages.append({"role": "user", "content": "Please respond ONLY with a JSON object."})
                continue
            
            data = json.loads(match.group(0))

            # Handle Tool Calls
            if "tool" in data:
                t_name = str(data.get("tool", "")).lower()
                args = data.get("args", {})
                
                u_name = args.get("name") or args.get("traveler_name") or "Guest"
                u_src = args.get("source") or args.get("src")
                u_dst = args.get("destination") or args.get("dst")
                
                obs = book_ticket(u_name, u_src, u_dst)
                thought_log.append(f"🛠️ Step {i+1} ACTION: Calling {t_name} -> Result: {obs}")
                
                if "SUCCESS" in obs:
                    thought_log.append("✅ Booking confirmed. Closing loop.")
                    return obs, "\n".join(thought_log)

                messages.append({"role": "assistant", "content": reply})
                messages.append({"role": "user", "content": f"Observation: {obs}"})
                continue

            if "final" in data: 
                thought_log.append(f"🏁 Step {i+1}: Final Answer reached.")
                return data["final"], "\n".join(thought_log)

        except Exception as e:
            thought_log.append(f"❌ Step {i+1} PARSE ERROR: {str(e)}")
            messages.append({"role": "user", "content": f"JSON error: {str(e)}. Fix your syntax."})
    
    return "I couldn't complete the booking.", "\n".join(thought_log)

In [ ]:
def chat_fn(message, history):
    answer, logs = agent_with_logs(message)
    history.append((message, answer))
    return "", history, logs

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ✈️ AI Ticket Agent with Live Logs")
    
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=400)
            msg = gr.Textbox(placeholder="Book Jon from Mumbai to Tokyo", label="Your Request")
        with gr.Column(scale=1):
            log_box = gr.Textbox(label="Agent Thought Process", interactive=False, lines=15)

    msg.submit(chat_fn, [msg, chatbot], [msg, chatbot, log_box])

demo.launch()